# Questão 1 - Expressão Matemática

Dada a seguinte expressão matemática:

$$
\frac{((5 + 4) * (8 - 3))}{3}
$$

Crie uma estrutura para representar e resolver essa expressão.

### O que deve ser feito

- Identifique qual o melhor tipo de estrutura para esse problema e justifique sua resposta.
- Identifique qual a melhor estratégia que deve ser utilizada para resolver esse problema.
- Crie e imprima a estrutura utilizada.
- Realize o cálculo provido utilizando a estrutura criada e o percurso identificado.
- Torne o seu código genérico, capaz de aceitar qualquer equação matemática no formato de string.

> **Importante:** podem considerar apenas as quatro operações básicas (`+`, `-`, `*`, `/`) e o uso dos parênteses. Ainda, não esqueçam das ordens de precedência das operações.

Apresente a análise de complexidade do algoritmo
implementado para resolver a questão.


## Resolução


Para resolver este problema utilizaremos uma **árvore binária de expressão**.

Nessa árvore, os números ficam nas folhas e os operadores ficam nos nós internos. Assim, cada operador aponta para os dois valores que ele precisa usar: o valor da esquerda e o valor da direita.


#### 1. Classe No e prioridade dos operadores

A classe `No` representa cada elemento da árvore. Um nó pode guardar um número ou um operador.

Também criamos o dicionário `PRIORIDADE` para indicar quais operações devem ser resolvidas primeiro. Multiplicação e divisão têm prioridade maior que soma e subtração.


In [1]:
PRIORIDADE = {
    '+': 1,
    '-': 1,
    '*': 2,
    '/': 2,
}


class No:
    def __init__(self, valor, esquerda=None, direita=None):
        self.valor = valor
        self.esquerda = esquerda
        self.direita = direita

    def eh_operador(self):
        return self.valor in PRIORIDADE


#### 2. Separando a expressão

A função `separar` recebe a expressão em formato de texto e divide tudo em partes menores: números, operadores e parênteses.

Também foram criadas duas funções auxiliares: `ajustar_simbolos`, para trocar símbolos matemáticos por símbolos usados no Python, e `ler_numero`, para converter textos numéricos em `int` ou `float`.


In [2]:
def ajustar_simbolos(expressao):
    return (
        expressao
        .replace('∗', '*')
        .replace('×', '*')
        .replace('−', '-')
        .replace('÷', '/')
    )


def ler_numero(texto):
    numero = float(texto)
    if numero.is_integer():
        return int(numero)
    return numero


def separar(expressao):
    expressao = ''.join(ajustar_simbolos(expressao).split())
    partes = []
    i = 0

    while i < len(expressao):
        char = expressao[i]
        menos_unario = (
            char == '-'
            and (not partes or partes[-1] in PRIORIDADE or partes[-1] == '(')
        )

        if char.isdigit() or char == '.' or menos_unario:
            if menos_unario and i + 1 < len(expressao) and expressao[i + 1] == '(':
                partes.append(0)
                partes.append('-')
                i += 1
                continue

            inicio = i
            if menos_unario:
                i += 1

            encontrou_digito = False
            pontos = 0
            while i < len(expressao) and (expressao[i].isdigit() or expressao[i] == '.'):
                if expressao[i].isdigit():
                    encontrou_digito = True
                elif expressao[i] == '.':
                    pontos += 1
                    if pontos > 1:
                        raise ValueError('Número inválido na expressão.')
                i += 1

            if not encontrou_digito:
                raise ValueError('Número inválido na expressão.')

            partes.append(ler_numero(expressao[inicio:i]))
            continue

        if char in PRIORIDADE or char in '()':
            partes.append(char)
            i += 1
            continue

        raise ValueError(f'Caractere inválido: {char}')

    return partes


#### 3. Transformando para pós-fixa

A expressão digitada normalmente está em forma infixa, como `5 + 4`.

Para facilitar a montagem da árvore, transformamos a expressão para a forma pós-fixa. Na pós-fixa, o operador aparece depois dos seus dois valores. Por exemplo:

`5 + 4` vira `5 4 +`


In [3]:
def para_posfixa(partes):
    posfixa = []
    pilha_operadores = []

    for parte in partes:
        if isinstance(parte, (int, float)):
            posfixa.append(parte)
        elif parte in PRIORIDADE:
            while (
                pilha_operadores
                and pilha_operadores[-1] in PRIORIDADE
                and PRIORIDADE[pilha_operadores[-1]] >= PRIORIDADE[parte]
            ):
                posfixa.append(pilha_operadores.pop())
            pilha_operadores.append(parte)
        elif parte == '(':
            pilha_operadores.append(parte)
        elif parte == ')':
            while pilha_operadores and pilha_operadores[-1] != '(':
                posfixa.append(pilha_operadores.pop())

            if not pilha_operadores:
                raise ValueError('Parênteses desbalanceados.')

            pilha_operadores.pop()

    while pilha_operadores:
        if pilha_operadores[-1] in '()':
            raise ValueError('Parênteses desbalanceados.')
        posfixa.append(pilha_operadores.pop())

    return posfixa


#### 4. Montando a árvore

Com a expressão em pós-fixa, montamos a árvore usando uma pilha.

Quando aparece um número, criamos um nó folha. Quando aparece um operador, pegamos os dois últimos nós criados e colocamos esse operador acima deles.


In [4]:
def montar_arvore(posfixa):
    pilha_nos = []

    for parte in posfixa:
        if isinstance(parte, (int, float)):
            pilha_nos.append(No(parte))
        elif parte in PRIORIDADE:
            if len(pilha_nos) < 2:
                raise ValueError('Expressão inválida.')

            direita = pilha_nos.pop()
            esquerda = pilha_nos.pop()
            pilha_nos.append(No(parte, esquerda, direita))

    if len(pilha_nos) != 1:
        raise ValueError('Expressão inválida.')

    return pilha_nos[0]


def criar_arvore(expressao):
    partes = separar(expressao)
    posfixa = para_posfixa(partes)
    return montar_arvore(posfixa)


#### 5. Percurso pós-ordem e cálculo

Para calcular o resultado, usamos o percurso pós-ordem.

Nesse percurso, primeiro calculamos o lado esquerdo, depois o lado direito e só então aplicamos o operador. Isso combina com a forma como uma expressão matemática precisa ser resolvida.


In [5]:
def pos_ordem(no):
    if no is None:
        return []

    return (
        pos_ordem(no.esquerda)
        + pos_ordem(no.direita)
        + [no.valor]
    )


def calcular(no):
    if not no.eh_operador():
        return no.valor

    esquerda = calcular(no.esquerda)
    direita = calcular(no.direita)

    if no.valor == '+':
        return esquerda + direita
    if no.valor == '-':
        return esquerda - direita
    if no.valor == '*':
        return esquerda * direita
    if no.valor == '/':
        return esquerda / direita

    raise ValueError('Operador inválido.')


#### 6. Mostrando a estrutura e executando

Por fim, criamos uma função para mostrar a árvore, exibir o percurso pós-ordem e apresentar o resultado final da expressão.


In [6]:
def formatar(valor):
    if isinstance(valor, float) and valor.is_integer():
        return str(int(valor))
    return str(valor)


def mostrar_arvore(no, nivel=0, lado='Raiz'):
    espacos = '    ' * nivel
    print(f'{espacos}{lado}: {formatar(no.valor)}')

    if no.esquerda is not None:
        mostrar_arvore(no.esquerda, nivel + 1, 'E')
    if no.direita is not None:
        mostrar_arvore(no.direita, nivel + 1, 'D')


def resolver(expressao):
    arvore = criar_arvore(expressao)
    percurso = pos_ordem(arvore)
    resultado = calcular(arvore)

    print('Expressão:', expressao)
    print('\nÁrvore binária de expressão:')
    mostrar_arvore(arvore)
    print('\nPercurso pós-ordem:', ' '.join(formatar(valor) for valor in percurso))
    print('Resultado:', formatar(resultado))

    return resultado


resolver('((5 + 4) * (8 - 3)) / 3')


Expressão: ((5 + 4) * (8 - 3)) / 3

Árvore binária de expressão:
Raiz: /
    E: *
        E: +
            E: 5
            D: 4
        D: -
            E: 8
            D: 3
    D: 3

Percurso pós-ordem: 5 4 + 8 3 - * 3 /
Resultado: 15


15.0

## Complexidade de algoritmos


Análise assintótica do algoritmo implementado:

- **Separação da expressão**: percorre o texto da expressão uma vez para identificar números, operadores e parênteses. Por isso, essa etapa tem tempo **O(n)**.

- **Conversão para pós-fixa**: cada parte da expressão entra e sai da pilha no máximo uma vez. Assim, essa etapa também tem tempo **O(n)**.

- **Montagem da árvore**: cada número e cada operador vira um nó ou é usado para ligar dois nós. Como cada parte é processada uma vez, essa etapa tem tempo **O(n)**.

- **Percurso pós-ordem e cálculo**: cada nó da árvore é visitado uma vez. Portanto, o cálculo também tem tempo **O(n)**.

O espaço utilizado é **O(n)**, pois a árvore guarda os elementos da expressão e as pilhas auxiliares também podem crescer de acordo com o tamanho da entrada.

Dessa forma, o algoritmo possui **tempo O(n)** e **espaço O(n)**.
